# Stone Soup walkthrough: filters, motion models and data association

This notebook builds trackers out of [Stone Soup](https://stonesoup.readthedocs.io)
components step by step, in the style of Stone Soup's own tutorials, and runs
them on this project's data: ADS-B ground truth and the simulated 2D radar's
detections, with their noise, slant range, unresolved targets and false
alarms. It starts with one aircraft and one filter, then changes one thing at
a time:

1. **The data**: truth and the radar's detections, simulated now or loaded
   from a stored run.
2. **One aircraft** to follow.
3. **Models**: how the target moves and what the radar measures.
4. **Extended Kalman filter**, and why a 2D radar makes height awkward.
5. **Unscented Kalman filter**.
6. **Particle filter**.
7. **Motion models**: constant acceleration, Singer, coordinated turn, and a
   multiple-model particle filter.
8. **Smoothing** a finished track.
9. **Replay** one aircraft: truth, detections and every filter, scan by scan.
10. **Many aircraft**: nearest neighbour and global nearest neighbour, with
    track initiation and deletion.
11. **Probabilistic association**: PDA and JPDA.
12. **Multi-frame assignment** (MFA), a multiple-hypothesis method.
13. **Comparison** of every tracker on the same detections.

Each step starts with a short **parameters** cell. Change a value and re-run
that step and the ones after it (or *Run > Run Selected Cell and All Below*).
Section 9's replay has its own controls: pick an aircraft, then play or drag
through the scans.

Needs the `notebook` extra (`uv sync --extra notebook`). Environment variables,
read once at the start:

| Variable | Meaning |
| --- | --- |
| `TRACKING_SIM_RUN` | A stored run's id (or unique prefix) to load instead of simulating |
| `TRACKING_SIM_RUNS` | Run store directory (default `runs`) |
| `TRACKING_SIM_RECORDING` | Truth to simulate on: a recording or SBS capture (default: the synthetic hour) |
| `TRACKING_SIM_WINDOW` | `START,END` in ISO 8601 UTC, part of the recording |
| `TRACKING_SIM_SBS_TZ` | Time zone of SBS timestamps (default UTC) |
| `TRACKING_SIM_SENSOR` | Radar spec YAML (default `scenarios/sensors/radar-1.yaml`) |
| `TRACKING_SIM_SEED` | Seed for the radar and the particle filters (default 42) |
| `TRACKING_SIM_AIRCRAFT` | Segment id of the aircraft to follow (default: the most detected) |

Relative paths are from the repository root.

In [ ]:
import os
import time
import warnings
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from tracking_sim.cli import parse_time
from tracking_sim.geo import LocalFrame
from tracking_sim.sensors import load_sensor_spec, parse_sensor_spec
from tracking_sim.sim.dataset import load_run, simulate, tracks_following
from tracking_sim.truth import build_truth
from tracking_sim.truth.sources import open_recording
from tracking_sim.viz.replay import SERIES, Estimate, Replay, horizontal_error

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def from_root(name):
    # A path from an environment variable; relative paths are from the repository root.
    value = os.environ.get(name)
    return str(ROOT / value) if value else None


INK, GREY, LIGHT = "#52514e", "#8a8984", "#c9c8c2"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. The data

A tracker reads **detections**: what the radar reported, scan by scan. Here
they come from this project's radar simulation, so every detection knows the
aircraft it came from (Stone Soup's `TrueDetection`) or that it is a false
alarm (`Clutter`), and every aircraft's **ground truth** is known, which is
what lets us measure how well each filter does.

Either simulate now (`RUN = None`): replay the truth past a radar spec, with
optional changes to the spec in `SENSOR_CHANGES`. Or load a stored run
(`RUN = "<run id>"`, from `tracking-sim runs list`): the detections are read
back exactly as that run's trackers saw them, and the run's own tracks come
along for comparison in section 13.

In [ ]:
# --- parameters ---
RUN = os.environ.get("TRACKING_SIM_RUN")  # a stored run id or prefix; None to simulate
RUNS = from_root("TRACKING_SIM_RUNS") or str(ROOT / "runs")
RECORDING = from_root("TRACKING_SIM_RECORDING") or str(ROOT / "tests/data/synthetic_1h.parquet")
WINDOW = os.environ.get("TRACKING_SIM_WINDOW")  # "START,END", or None for all of it
SBS_TZ = os.environ.get("TRACKING_SIM_SBS_TZ", "UTC")
SENSOR = from_root("TRACKING_SIM_SENSOR") or str(ROOT / "scenarios/sensors/radar-1.yaml")
SEED = int(os.environ.get("TRACKING_SIM_SEED", "42"))
# Changes to the radar spec, merged section by section, for example
# {"noise": {"range_sigma_m": 150}, "detection": {"pd": 0.7, "false_alarm_density": 5e-9}}
SENSOR_CHANGES = {}

In [ ]:
if RUN:
    data = load_run(RUN, store=RUNS)
else:
    spec_data = load_sensor_spec(SENSOR).model_dump()
    for section, changes in SENSOR_CHANGES.items():
        spec_data[section] = (
            {**spec_data[section], **changes} if isinstance(changes, dict) else changes
        )
    radar_spec = parse_sensor_spec(spec_data)
    pos = radar_spec.position
    frame = LocalFrame(pos.lat, pos.lon, pos.alt_m)  # scenario frame centred on the radar
    window = [parse_time(t) for t in WINDOW.split(",")] if WINDOW else [None, None]
    records = list(open_recording(RECORDING, receiver="truth", tz=SBS_TZ).replay(*window))
    truth = build_truth(records, frame)
    data = simulate(truth, radar_spec, seed=SEED)

radar = data.sensor.spec
print(data)
print(
    f"radar {radar.id}: scan {data.period_s:g} s, noise {radar.noise.range_sigma_m:g} m in range, "
    f"{radar.noise.bearing_sigma_deg:g} deg in bearing; Pd {data.pd}; "
    f"{data.false_alarms_per_scan:.1f} false alarms a scan"
)
aircraft = data.aircraft().to_pandas()
aircraft.head(10)

## 2. One aircraft

The table above lists every aircraft the radar detected, most detections
first, with how hard it turned while detected (`turn_deg_s`). Pick one to
follow through sections 3 to 9: a manoeuvring one shows the differences
between motion models best.

The left plot is what the radar saw: each detection placed where a radar
display puts it (slant range laid along the ground), over the true path. The
right plots are the measurement errors: detection minus truth, in the two
things a 2D radar measures. They are the radar's noise, plus a range
offset that grows with height: slant range is longer than ground range.

In [ ]:
# --- parameters ---
AIRCRAFT = os.environ.get("TRACKING_SIM_AIRCRAFT") or aircraft.segment_id.iloc[0]

In [ ]:
detections = data.detections_of(AIRCRAFT)
row = aircraft.set_index("segment_id").loc[AIRCRAFT]
print(
    f"{AIRCRAFT} ({row.callsign}): {len(detections)} detections over {row.minutes:.1f} min, "
    f"{row.height_m:.0f} m above the scenario origin, {row.speed_mps:.0f} m/s, "
    f"turning up to {row.turn_deg_s:.1f} deg/s"
)

segment = data.truth.segment(AIRCRAFT)
t_det = np.array([d.timestamp.timestamp() for d in detections])
truth_pos, _, _ = segment.evaluate(np.arange(t_det[0], t_det[-1], 1.0))

# What a perfect radar would have measured: the truth seen from the antenna.
true_az, _, true_range = data.sensor.site.polar(segment.evaluate(t_det)[0])
bearing = np.array([float(d.state_vector[0, 0]) for d in detections])
measured_az = np.mod(np.pi / 2 - bearing, 2 * np.pi)  # Stone Soup bearing to compass azimuth
measured_range = np.array([float(d.state_vector[1, 0]) for d in detections])
az_error = np.degrees((measured_az - true_az + np.pi) % (2 * np.pi) - np.pi)
range_error = measured_range - true_range

fig = plt.figure(figsize=(11, 4.6))
grid = fig.add_gridspec(2, 2, width_ratios=[1.1, 1])
ax = fig.add_subplot(grid[:, 0])
ax.plot(*(truth_pos[:, :2] / 1e3).T, color=INK, lw=1.5, label="truth")
ax.scatter(
    *(data.plot_positions(detections) / 1e3).T,
    s=8,
    color=SERIES[0],
    label="detections (radar display)",
)
ax.set(xlabel="east (km)", ylabel="north (km)", aspect="equal")
ax.legend(frameon=False, fontsize=8)
ax.set_title(f"{AIRCRAFT} {row.callsign}", loc="left", fontsize=10)
minutes = (t_det - t_det[0]) / 60
for k, (values, label) in enumerate(
    [(az_error, "bearing error (deg)"), (range_error, "slant range error (m)")]
):
    a = fig.add_subplot(grid[k, 1])
    a.scatter(minutes, values, s=6, color=SERIES[0])
    a.axhline(0, color=GREY, lw=1)
    a.set_ylabel(label)
a.set_xlabel("minutes since first detection")
fig.tight_layout()
plt.show()
print(
    f"bearing error sd {az_error.std():.2f} deg (spec {radar.noise.bearing_sigma_deg:g}); "
    f"range error sd {range_error.std():.0f} m (spec {radar.noise.range_sigma_m:g})"
)

## 3. Models

Every Stone Soup filter needs two models.

**Transition model**: how the state moves between measurements. The
simplest is *constant velocity* on each axis: the state is
`[x, vx, y, vy, z, vz]` (east, north, up in metres, and their rates), and the
aircraft keeps its velocity apart from random accelerations of strength `Q`
(Stone Soup's `noise_diff_coeff`, in m^2/s^3). A bigger `Q` lets the filter
follow manoeuvres but trust its prediction less.

**Measurement model**: what the radar reports for a given state. Here it is
`CartesianToBearingSlantRange`: bearing and the straight-line range from the
antenna, with the radar's noise. Every detection already carries it, built
for the radar's site, so the filters below never need it spelled out.

**A 2D radar measures no height.** Height only enters through slant range,
and only weakly. A new track therefore starts at an *assumed* height with a
large uncertainty, and the filter is told height changes slowly: small
vertical noise `QZ` and small vertical speed spread. With more freedom it can
explain range errors by moving the aircraft up or down, which section 4
shows.

**The prior** is the filter's starting belief, made from the first detection:
position from bearing and slant range at the assumed height (with the
height's uncertainty carried into ground range), velocity zero with a wide
spread.

In [ ]:
# --- parameters ---
Q = 5.0  # horizontal acceleration noise, m^2/s^3
QZ = 0.01  # vertical acceleration noise, m^2/s^3
ASSUMED_HEIGHT_M = 5000  # starting height above the antenna
HEIGHT_SIGMA_M = 3000
SPEED_SIGMA_MPS = 150  # prior horizontal speed spread
VERTICAL_SPEED_SIGMA_MPS = 1.0

In [ ]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.state import GaussianState
from stonesoup.types.track import Track

from tracking_sim.tracking.adapt import DetectionAdapter
from tracking_sim.tracking.initiators import initial_position

constant_velocity = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(Q), ConstantVelocity(Q), ConstantVelocity(QZ)]
)
print("measurement model:", type(data.measurement_model).__name__)
print("noise covariance (rad^2, m^2):", np.diag(data.measurement_model.covar()))


def make_prior(detection, ndim=6, position=(0, 2, 4), velocity=(1, 3, 5), extra=()):
    """A Gaussian prior from one detection, for a state of any layout.

    ``extra`` adds (index, variance) pairs for other state entries, such as
    acceleration or turn rate.
    """
    bearing, slant = float(detection.state_vector[0, 0]), float(detection.state_vector[1, 0])
    pos, pos_cov = initial_position(
        detection.measurement_model, bearing, slant, ASSUMED_HEIGHT_M, HEIGHT_SIGMA_M
    )
    mean, covar = np.zeros(ndim), np.zeros((ndim, ndim))
    mean[list(position)] = pos
    covar[np.ix_(position, position)] = pos_cov
    covar[velocity[0], velocity[0]] = covar[velocity[1], velocity[1]] = SPEED_SIGMA_MPS**2
    covar[velocity[2], velocity[2]] = VERTICAL_SPEED_SIGMA_MPS**2
    for index, variance in extra:
        covar[index, index] = variance
    return GaussianState(mean.reshape(-1, 1), covar, timestamp=detection.timestamp)


def run_filter(predictor, updater, prior, detections):
    """The Stone Soup filtering loop: predict to each detection, then update."""
    track = Track([prior])
    for detection in detections:
        prediction = predictor.predict(track[-1], timestamp=detection.timestamp)
        hypothesis = SingleHypothesis(prediction, detection)
        track.append(updater.update(hypothesis))
    return track


def error_summary(estimate):
    """Median and 95th percentile horizontal error, after the first fifth of the flight."""
    _, err = horizontal_error(data, AIRCRAFT, estimate)
    settled = err[len(err) // 5 :]
    return np.nanmedian(settled), np.nanpercentile(settled, 95)


results = {}  # label -> (Estimate, seconds)


def report(label, track, seconds, position=(0, 2, 4)):
    estimate = Estimate(label, track, position)
    results[label] = (estimate, seconds)
    median, p95 = error_summary(estimate)
    print(f"{label}: median error {median:.0f} m, 95th percentile {p95:.0f} m, {seconds:.2f} s")
    return estimate


def plot_errors(labels, title="horizontal position error against truth"):
    _, ax = plt.subplots(figsize=(9, 3.4))
    for i, label in enumerate(labels):
        t, err = horizontal_error(data, AIRCRAFT, results[label][0])
        ax.plot((t - t[0]) / 60, err, color=SERIES[i], lw=1.5, label=label)
    ax.set(xlabel="minutes since first detection", ylabel="error (m)", ylim=(0, None))
    errors = [horizontal_error(data, AIRCRAFT, results[name][0])[1] for name in labels]
    # Scale to the typical errors, so one diverging filter can't flatten the rest.
    top = min(
        np.nanpercentile(np.concatenate(errors), 98), 6 * max(np.nanmedian(e) for e in errors)
    )
    ax.set_ylim(0, top * 1.15)
    ax.legend(frameon=False, fontsize=8, ncols=min(len(labels), 4))
    ax.set_title(title, loc="left", fontsize=10)
    plt.show()

## 4. Extended Kalman filter

The radar's measurement is a non-linear function of position (an angle and
a distance), so the plain Kalman filter does not apply. The **extended Kalman
filter** (EKF) linearises the measurement model around each prediction. The
predictor applies the transition model; the updater blends the prediction
with the detection, each weighted by its uncertainty.

The error plot shows the filter settling in over the first scans (the prior
knew nothing about velocity), then following the aircraft. The grey band is
the filter's own 2-sigma estimate of its horizontal error: a consistent
filter's error stays mostly inside it.

In [ ]:
from stonesoup.predictor.kalman import ExtendedKalmanPredictor
from stonesoup.updater.kalman import ExtendedKalmanUpdater

ekf_predictor = ExtendedKalmanPredictor(constant_velocity)
ekf_updater = ExtendedKalmanUpdater(measurement_model=None)  # each detection brings its own

started = time.perf_counter()
ekf_track = run_filter(ekf_predictor, ekf_updater, make_prior(detections[0]), detections[1:])
ekf = report("EKF", ekf_track, time.perf_counter() - started)

t, err = horizontal_error(data, AIRCRAFT, ekf)
sigma = np.array([np.sqrt(s.covar[0, 0] + s.covar[2, 2]) for s in ekf_track.states])
fig, ax = plt.subplots(figsize=(9, 3.4))
minutes = (t - t[0]) / 60
ax.fill_between(minutes, 0, 2 * sigma, color=LIGHT, alpha=0.6, lw=0, label="filter's 2-sigma")
ax.plot(minutes, err, color=SERIES[0], lw=1.5, label="EKF error")
ax.set(
    xlabel="minutes since first detection",
    ylabel="error (m)",
    ylim=(0, np.nanpercentile(err, 98) * 1.3),
)
ax.legend(frameon=False, fontsize=8)
plt.show()
print(f"error inside the 2-sigma band {np.mean(err <= 2 * sigma):.0%} of the time")

**Why height noise matters.** Re-run the same filter with the vertical
settings the milestone 3 trackers use (`QZ = 0.1`, vertical speed spread
10 m/s). Free to change height, the filter can explain slant-range errors by
climbing or descending, and when the aircraft is far from the assumed height
its ground position drifts with it.

In [ ]:
loose_motion = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(Q), ConstantVelocity(Q), ConstantVelocity(0.1)]
)
loose_prior = make_prior(detections[0])
loose_prior.covar[5, 5] = 10.0**2
loose = run_filter(ExtendedKalmanPredictor(loose_motion), ekf_updater, loose_prior, detections[1:])
heights = np.array([float(s.state_vector[4, 0]) for s in loose.states])
median, p95 = error_summary(Estimate("loose", loose))
print(
    f"EKF, loose height: median error {median:.0f} m, 95th percentile {p95:.0f} m; "
    f"its height wandered from {heights.min():.0f} to {heights.max():.0f} m "
    f"(truth about {row.height_m:.0f} m)"
)

## 5. Unscented Kalman filter

The **unscented Kalman filter** (UKF) does not linearise. It pushes a small
set of *sigma points* spread around the estimate through the models and
re-fits a Gaussian to where they land. That captures more of the
non-linearity, and it needs no derivatives, but it takes the uncertainty at
face value: with 3 km of height uncertainty, the sigma points sample a wide
range of heights, and each height means a different ground range for the
same slant range. `ALPHA`, `BETA` and `KAPPA` set how far the points spread
and how they are weighted (Stone Soup's defaults: 0.5, 2 and `3 - n`).

In [ ]:
# --- parameters ---
ALPHA, BETA, KAPPA = 0.5, 2.0, None  # None: Stone Soup's default, 3 - state size

In [ ]:
from stonesoup.predictor.kalman import UnscentedKalmanPredictor
from stonesoup.updater.kalman import UnscentedKalmanUpdater

sigma_points = {"alpha": ALPHA, "beta": BETA, "kappa": KAPPA}
ukf_predictor = UnscentedKalmanPredictor(constant_velocity, **sigma_points)
ukf_updater = UnscentedKalmanUpdater(measurement_model=None, **sigma_points)

started = time.perf_counter()
ukf_track = run_filter(ukf_predictor, ukf_updater, make_prior(detections[0]), detections[1:])
report("UKF", ukf_track, time.perf_counter() - started)
plot_errors(["EKF", "UKF"])

## 6. Particle filter

A **particle filter** represents the belief by many samples (*particles*)
instead of a Gaussian: each is moved by the transition model with its own
random noise, weighted by how well it explains the detection, and the set is
resampled when too few carry the weight. It copes with any model and any
shape of uncertainty, at the cost of many more calculations.

Particles need a decent start. Spread over the first detection's prior
(position uncertain by kilometres, velocity unknown), most land somewhere
useless and the filter can lose the aircraft. So, like this project's
particle tracker, start them from the EKF's estimate after `WARM_START`
detections. Set `WARM_START = 0` to see the difference.

In [ ]:
# --- parameters ---
PARTICLES = 1000
WARM_START = 3  # detections handled by the EKF before switching to particles

In [ ]:
from stonesoup.predictor.particle import ParticlePredictor
from stonesoup.resampler.particle import ESSResampler
from stonesoup.types.array import StateVectors
from stonesoup.types.state import ParticleState
from stonesoup.updater.particle import ParticleUpdater


def particles_from(gaussian, n, rng):
    samples = rng.multivariate_normal(np.ravel(gaussian.state_vector), gaussian.covar, n).T
    return ParticleState(
        StateVectors(samples), log_weight=np.full(n, -np.log(n)), timestamp=gaussian.timestamp
    )


rng = np.random.default_rng(SEED)
np.random.seed(SEED)  # noqa: NPY002 (Stone Soup's particle noise and resampling use NumPy's global generator)
start_state = ekf_track.states[WARM_START]
pf_predictor = ParticlePredictor(constant_velocity)
pf_updater = ParticleUpdater(measurement_model=None, resampler=ESSResampler())

started = time.perf_counter()
pf_track = run_filter(
    pf_predictor,
    pf_updater,
    particles_from(start_state, PARTICLES, rng),
    detections[WARM_START + 1 :],
)
report("PF", pf_track, time.perf_counter() - started)
plot_errors(["EKF", "UKF", "PF"])

## 7. Motion models

Constant velocity assumes straight flight disturbed by random pushes. Other
models assume more about how aircraft move, which needs a different state:

| Model | State | Assumes |
| --- | --- | --- |
| Constant acceleration | `[x, vx, ax, y, vy, ay, z, vz]` | acceleration persists, disturbed by random jerk |
| Singer | same | acceleration decays back to zero with time constant `SINGER_TAU_S` |
| Coordinated turn | `[x, vx, y, vy, w, z, vz]` | a turn at rate `w` (rad/s), which changes slowly |
| Multiple model | coordinated turn's | particles switch between straight flight and turning |

Height keeps the constant-velocity model throughout: a 2D radar cannot see
vertical acceleration at all.

The detections carry a measurement model for the 6D state; a filter with a
different state needs the same model mapped to its own layout, which
`DetectionAdapter` does (it copies each detection with a remapped model).
The coordinated turn is non-linear in the state too, so it runs on both the
EKF and the UKF. The EKF linearises the turn around its current turn-rate
estimate; give the turn rate too much freedom (`TURN_NOISE_DEG = 1.0`, as the
particle tracker uses) and on a manoeuvring aircraft that estimate can run
away, and the track with it. The UKF and the particles are more forgiving.

In [ ]:
# --- parameters ---
CA_NOISE = 0.5  # constant acceleration: jerk noise, m^2/s^5
SINGER_NOISE = 1.0  # Singer: acceleration noise
SINGER_TAU_S = 10.0  # Singer: how long an acceleration lasts, s
ACCEL_SIGMA = 3.0  # prior acceleration spread, m/s^2
TURN_NOISE_DEG = 0.1  # coordinated turn: turn-rate noise, deg^2/s^3 (try 1.0 with the EKF)
TURN_SIGMA_DEG = 3.0  # prior turn-rate spread, deg/s
SWITCH_PROBABILITY = 0.05  # multiple model: chance of switching model per update

In [ ]:
from stonesoup.models.transition.base import CombinedGaussianTransitionModel
from stonesoup.models.transition.linear import ConstantAcceleration, Singer
from stonesoup.models.transition.nonlinear import ConstantTurn
from stonesoup.predictor.particle import MultiModelPredictor
from stonesoup.types.state import MultiModelParticleState
from stonesoup.updater.particle import MultiModelParticleUpdater

# Constant acceleration and Singer: 8D, positions at 0, 3, 6.
to_8d = DetectionAdapter(8, (0, 3, 6))
detections_8d = [to_8d.one(d) for d in detections]
accel_prior = make_prior(
    detections_8d[0], 8, (0, 3, 6), (1, 4, 7), extra=[(2, ACCEL_SIGMA**2), (5, ACCEL_SIGMA**2)]
)
for label, axis_model in (
    ("EKF CA", ConstantAcceleration(CA_NOISE)),
    ("EKF Singer", Singer(SINGER_NOISE, 1 / SINGER_TAU_S)),
):
    motion = CombinedLinearGaussianTransitionModel([axis_model, axis_model, ConstantVelocity(QZ)])
    started = time.perf_counter()
    track = run_filter(ExtendedKalmanPredictor(motion), ekf_updater, accel_prior, detections_8d[1:])
    report(label, track, time.perf_counter() - started, (0, 3, 6))

# Coordinated turn: 7D, positions at 0, 2, 5, turn rate at 4.
to_7d = DetectionAdapter(7, (0, 2, 5))
detections_7d = [to_7d.one(d) for d in detections]
turn_noise = np.radians(np.sqrt(TURN_NOISE_DEG)) ** 2
turning = CombinedGaussianTransitionModel([ConstantTurn([Q, Q], turn_noise), ConstantVelocity(QZ)])
turn_prior = make_prior(
    detections_7d[0], 7, (0, 2, 5), (1, 3, 6), extra=[(4, np.radians(TURN_SIGMA_DEG) ** 2)]
)
for label, predictor, updater in (
    ("EKF turn", ExtendedKalmanPredictor(turning), ekf_updater),
    ("UKF turn", UnscentedKalmanPredictor(turning, **sigma_points), ukf_updater),
):
    started = time.perf_counter()
    track = run_filter(predictor, updater, turn_prior, detections_7d[1:])
    report(label, track, time.perf_counter() - started, (0, 2, 5))

# Multiple model: each particle flies straight (model 0) or turns (model 1).
straight = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(Q), ConstantVelocity(Q), ConstantVelocity(QZ)]
)
p = SWITCH_PROBABILITY
mm_predictor = MultiModelPredictor(
    transition_models=[straight, turning],
    transition_matrix=np.array([[1 - p, p], [p, 1 - p]]),
    model_mappings=[(0, 1, 2, 3, 5, 6), (0, 1, 2, 3, 4, 5, 6)],  # straight flight ignores w
)
mm_updater = MultiModelParticleUpdater(
    measurement_model=None, predictor=mm_predictor, resampler=ESSResampler()
)


def mm_particles_from(gaussian, n, rng):
    """Multiple-model particles from a 6D Gaussian, turn rate added, all flying straight."""
    mean = np.insert(np.ravel(gaussian.state_vector), 4, 0.0)
    covar = np.insert(np.insert(np.asarray(gaussian.covar), 4, 0.0, axis=0), 4, 0.0, axis=1)
    covar[4, 4] = np.radians(TURN_SIGMA_DEG) ** 2
    samples = rng.multivariate_normal(mean, covar, n).T
    return MultiModelParticleState(
        StateVectors(samples),
        log_weight=np.full(n, -np.log(n)),
        dynamic_model=np.zeros(n, dtype=int),
        timestamp=gaussian.timestamp,
    )


# Warm start from the EKF, like the particle filter above.
started = time.perf_counter()
track = run_filter(
    mm_predictor,
    mm_updater,
    mm_particles_from(ekf_track.states[WARM_START], PARTICLES, rng),
    detections_7d[WARM_START + 1 :],
)
report("PF multiple model", track, time.perf_counter() - started, (0, 2, 5))

plot_errors(
    ["EKF", "EKF CA", "EKF Singer", "EKF turn", "UKF turn", "PF multiple model"],
    "motion models: horizontal position error",
)

## 8. Smoothing

A filter only uses detections up to now. Once a track is finished, a
**smoother** runs back over it and lets every state benefit from the
detections after it too (Rauch-Tung-Striebel). It is no use for a live
display, but it is the best reconstruction of what happened, for example to
check a sensor against the track afterwards. Stone Soup's smoother needs the
same transition model the filter used.

In [ ]:
from stonesoup.smoother.kalman import ExtendedKalmanSmoother, UnscentedKalmanSmoother

started = time.perf_counter()
report(
    "EKF smoothed",
    ExtendedKalmanSmoother(constant_velocity).smooth(ekf_track),
    time.perf_counter() - started,
)
started = time.perf_counter()
report(
    "UKF smoothed",
    UnscentedKalmanSmoother(constant_velocity).smooth(ukf_track),
    time.perf_counter() - started,
)
plot_errors(["EKF", "EKF smoothed", "UKF", "UKF smoothed"], "filtered and smoothed")

## 9. Replay one aircraft

Truth, detections and filters together, scan by scan. Press play, or drag
the scan slider. The map shows the aircraft's true path (grey dashes ahead,
solid behind), its detections so far, the rest of the radar's detections
from that scan (other aircraft and false alarms, as crosses), and each
filter's track with its 2-sigma position ellipse; the right panel is each
filter's error, with the current scan marked.

Pick another aircraft from the list to run the filters in `REPLAY_FILTERS` on
it with the parameters above (up to eight; their names are those in
`results`).

In [ ]:
# --- parameters ---
REPLAY_FILTERS = ["EKF", "UKF", "PF", "UKF turn", "PF multiple model", "EKF smoothed"]
REPLAY_AIRCRAFT = list(aircraft.segment_id.iloc[:25])  # offered in the picker

In [ ]:
def filters_for(segment_id):
    """Run the replay filters on one aircraft, with the parameters above."""
    dets = data.detections_of(segment_id)
    dets_7d = [to_7d.one(d) for d in dets]
    prior_7d = make_prior(
        dets_7d[0], 7, (0, 2, 5), (1, 3, 6), extra=[(4, np.radians(TURN_SIGMA_DEG) ** 2)]
    )
    local_rng = np.random.default_rng(SEED)
    np.random.seed(SEED)  # noqa: NPY002 (Stone Soup's particle filters)
    ekf_t = run_filter(ekf_predictor, ekf_updater, make_prior(dets[0]), dets[1:])
    ukf_t = run_filter(ukf_predictor, ukf_updater, make_prior(dets[0]), dets[1:])
    warm = len(dets) > WARM_START + 1
    makers = {
        "EKF": lambda: Estimate("EKF", ekf_t),
        "UKF": lambda: Estimate("UKF", ukf_t),
        "EKF smoothed": lambda: Estimate(
            "EKF smoothed", ExtendedKalmanSmoother(constant_velocity).smooth(ekf_t)
        ),
        "UKF smoothed": lambda: Estimate(
            "UKF smoothed", UnscentedKalmanSmoother(constant_velocity).smooth(ukf_t)
        ),
        "PF": lambda: Estimate(
            "PF",
            run_filter(
                pf_predictor,
                pf_updater,
                particles_from(ekf_t.states[WARM_START], PARTICLES, local_rng),
                dets[WARM_START + 1 :],
            ),
        ),
        "EKF turn": lambda: Estimate(
            "EKF turn",
            run_filter(ExtendedKalmanPredictor(turning), ekf_updater, prior_7d, dets_7d[1:]),
            (0, 2, 5),
        ),
        "UKF turn": lambda: Estimate(
            "UKF turn",
            run_filter(
                UnscentedKalmanPredictor(turning, **sigma_points),
                ukf_updater,
                prior_7d,
                dets_7d[1:],
            ),
            (0, 2, 5),
        ),
        "PF multiple model": lambda: Estimate(
            "PF multiple model",
            run_filter(
                mm_predictor,
                mm_updater,
                mm_particles_from(ekf_t.states[WARM_START], PARTICLES, local_rng),
                dets_7d[WARM_START + 1 :],
            ),
            (0, 2, 5),
        ),
    }
    needs_warm_start = {"PF", "PF multiple model"}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return [
            makers[name]()
            for name in REPLAY_FILTERS
            if name in makers and (warm or name not in needs_warm_start)
        ]


replay = Replay(data, AIRCRAFT, [results[name][0] for name in REPLAY_FILTERS])
player = replay.widget(
    filters_for, REPLAY_AIRCRAFT if AIRCRAFT in REPLAY_AIRCRAFT else [AIRCRAFT, *REPLAY_AIRCRAFT]
)
display(player)
print("final scan:", replay.summary(-1))

## 10. Many aircraft: data association

So far each filter was handed exactly its aircraft's detections. A real
tracker gets every detection of every scan, false alarms included, and has
to decide which detection belongs to which track (**data association**),
when a run of unexplained detections is a new aircraft (**initiation**),
and when a track has lost its aircraft (**deletion**).

The pieces, each a Stone Soup component:

* a **hypothesiser** scores every track-detection pair: here the Mahalanobis
  distance between the detection and the track's predicted measurement,
  with `GATE` as the distance beyond which a pair is not considered.
  `PreGatedHypothesiser` (this project's) throws out hopeless pairs cheaply
  first; it changes speed, not results.
* a **data associator** picks the pairs: *nearest neighbour* (NN) gives each
  track its closest detection, one track at a time; *global nearest
  neighbour* (GNN) finds the best assignment for all tracks together.
* an **initiator** starts tentative tracks from unassociated detections, at
  the assumed height, and confirms them after `MIN_POINTS` associated
  detections; tentative tracks that miss `TENTATIVE_MISSES` scans are dropped.
* a **deleter** drops a confirmed track after `DELETE_AFTER_MISSES` scans
  without an update.

The more careful associators get expensive with many tracks and detections,
so this section and the next work on a **subset**: `SUBSET_MINUTES` of
scans from the chosen aircraft's first detection, keeping only detections
(and truth) within `SUBSET_RADIUS_KM` of that aircraft as it flies. Set
`SUBSET_RADIUS_KM = None` for the whole coverage.

In [ ]:
# --- parameters ---
SUBSET_MINUTES = 10
SUBSET_RADIUS_KM = 40  # None: the whole coverage
GATE = 3.0  # Mahalanobis distance
MIN_POINTS = 3  # detections to confirm a track
TENTATIVE_MISSES = 2  # scans a tentative track may go without an update
DELETE_AFTER_MISSES = 3  # scans a confirmed track may go without an update

In [ ]:
from stonesoup.dataassociator.neighbour import GNNWith2DAssignment, NearestNeighbour
from stonesoup.deleter.time import UpdateTimeStepsDeleter
from stonesoup.hypothesiser.distance import DistanceHypothesiser
from stonesoup.initiator.simple import MultiMeasurementInitiator
from stonesoup.measures import Mahalanobis
from stonesoup.tracker.simple import MultiTargetTracker

from tracking_sim.tracking.gating import PreGatedHypothesiser
from tracking_sim.tracking.initiators import SlantRangeInitiator

first = data.detections_of(AIRCRAFT)[0].timestamp
area = dict(around=AIRCRAFT, radius_m=SUBSET_RADIUS_KM * 1e3) if SUBSET_RADIUS_KM else {}
part = data.subset(first, first + timedelta(minutes=SUBSET_MINUTES), **area)
clutter_per_scan = np.mean(
    [sum(type(d).__name__ == "Clutter" for d in s.detections) for s in part.scans]
)
print(part)
print(
    f"{len(part.looks)} aircraft visible, {clutter_per_scan:.1f} false alarms a scan in the subset"
)


def pre_gated(hypothesiser):
    return PreGatedHypothesiser(
        hypothesiser=hypothesiser, position_mapping=(0, 2, 4), velocity_mapping=(1, 3, 5)
    )


distance_hypothesiser = pre_gated(
    DistanceHypothesiser(ekf_predictor, ekf_updater, Mahalanobis(), missed_distance=GATE)
)
velocity_prior = GaussianState(
    np.zeros((6, 1)),
    np.diag([0, SPEED_SIGMA_MPS**2, 0, SPEED_SIGMA_MPS**2, 0, VERTICAL_SPEED_SIGMA_MPS**2]),
)


def make_initiator():
    return MultiMeasurementInitiator(
        prior_state=velocity_prior,
        deleter=UpdateTimeStepsDeleter(TENTATIVE_MISSES),
        data_associator=GNNWith2DAssignment(distance_hypothesiser),
        updater=ekf_updater,
        min_points=MIN_POINTS,
        initiator=SlantRangeInitiator(
            prior_state=velocity_prior, height_m=ASSUMED_HEIGHT_M, height_sigma_m=HEIGHT_SIGMA_M
        ),
    )


def run_tracker(tracker):
    """Run a Stone Soup tracker over the subset; keep every track it ever held."""
    held = {}
    started = time.perf_counter()
    for _, tracks in tracker:
        for track in tracks:
            held.setdefault(id(track), track)
    return list(held.values()), time.perf_counter() - started


trackers = {}  # name -> (tracks, seconds, position mapping)
for name, associator in (
    ("NN", NearestNeighbour(distance_hypothesiser)),
    ("GNN", GNNWith2DAssignment(distance_hypothesiser)),
):
    tracker = MultiTargetTracker(
        initiator=make_initiator(),
        deleter=UpdateTimeStepsDeleter(DELETE_AFTER_MISSES),
        detector=part.batches(),
        data_associator=associator,
        updater=ekf_updater,
    )
    tracks, seconds = run_tracker(tracker)
    trackers[name] = (tracks, seconds, (0, 2, 4))
    print(f"{name}: {len(tracks)} tracks in {seconds:.1f} s")

## 11. Probabilistic association: PDA and JPDA

Nearest-neighbour methods commit to one detection per track per scan; when
a false alarm sits closer than the true detection, the track takes the
wrong one. **Probabilistic data association** (PDA) hedges instead: it
updates the track with *every* detection in its gate, weighted by how likely
each is to be the aircraft's, including the chance that none is (missed
detection). The weights need the radar's probability of detection and the
false-alarm density, both of which the radar spec gives.

PDA treats each track on its own, so two tracks near each other can both
claim the same detection. **Joint PDA** (JPDA) weighs the joint events
across tracks, so each detection is shared out consistently. Its cost grows
quickly with the number of tracks competing for detections; `JPDAwithEHM`
computes the same answer faster with an efficient hypothesis management tree.

These trackers keep a Gaussian per track that blends several detections, so
they use Stone Soup's `MultiTargetMixtureTracker`.

`PROB_GATE` sets the gate: the chance the aircraft's own detection falls
inside it, if the track's uncertainty is right. A constant-velocity track
lags a turning aircraft by more than its uncertainty admits, so the gate is
set wide: 0.999999 is about 5.3 standard deviations. At 0.9999 (4.3) a track
on a turning aircraft can lose it; the trackers here start a new track, but
the multi-frame assignment in section 12 cannot.

In [ ]:
# --- parameters ---
PROB_GATE = 0.999999  # probability the true detection falls inside the gate
JPDA_METHOD = "JPDAwithEHM"  # or "JPDA"

In [ ]:
from stonesoup.dataassociator.probability import JPDA, PDA, JPDAwithEHM
from stonesoup.hypothesiser.probability import PDAHypothesiser
from stonesoup.tracker.simple import MultiTargetMixtureTracker

prob_detect = (
    data.pd if data.pd is not None else 0.8
)  # radar-equation Pd varies; use a typical value
clutter_density = data.clutter_density()
print(f"Pd {prob_detect}, clutter density {clutter_density:.3g} per rad per m per scan")
pda_hypothesiser = pre_gated(
    PDAHypothesiser(
        ekf_predictor,
        ekf_updater,
        clutter_spatial_density=clutter_density,
        prob_detect=prob_detect,
        prob_gate=PROB_GATE,
    )
)
jpda = {"JPDA": JPDA, "JPDAwithEHM": JPDAwithEHM}[JPDA_METHOD]
for name, associator in (("PDA", PDA(pda_hypothesiser)), ("JPDA", jpda(pda_hypothesiser))):
    tracker = MultiTargetMixtureTracker(
        initiator=make_initiator(),
        deleter=UpdateTimeStepsDeleter(DELETE_AFTER_MISSES),
        detector=part.batches(),
        data_associator=associator,
        updater=ekf_updater,
    )
    tracks, seconds = run_tracker(tracker)
    trackers[name] = (tracks, seconds, (0, 2, 4))
    print(f"{name}: {len(tracks)} tracks in {seconds:.1f} s")

## 12. Multi-frame assignment

Every method so far decides each scan's associations for good. A
**multiple-hypothesis** tracker keeps several association histories alive
and decides later, once more scans have shown which explanation holds up.
Stone Soup's **multi-frame assignment** (MFA) does this over a sliding window
of `SLIDE_WINDOW` scans: each track is a mixture of Gaussians, one per
surviving association history, and the histories that disagree with the
best assignment across the window are pruned.

Stone Soup's MFA has no track initiation or deletion of its own, so, as in
Stone Soup's MFA example, its tracks are **seeded from truth**: one per
aircraft in the subset, nearest the chosen one first. Each starts like a
confirmed track elsewhere: an EKF over the aircraft's first `MIN_POINTS`
detections in the subset, so it knows roughly how fast the aircraft is
going. (Seeded from one detection at zero velocity, a fast jet leaves the
gate before the first update.) Knowing which detections to start from makes
MFA better informed than the trackers above, which had to find the aircraft
among the false alarms; compare its association, not its initiation. It
also follows only the `MFA_MAX_AIRCRAFT` it was seeded with, so in busy
traffic its completeness counts the others as missed.
Detections within a scan have their own times, which Stone Soup's MFA
warns about; that is expected here.

MFA solves its assignment with Google OR-Tools (`ortools`, in the
`notebook` extra).

In [ ]:
# --- parameters ---
SLIDE_WINDOW = 3  # scans of association history MFA keeps open
MFA_MAX_AIRCRAFT = 10  # seed at most this many tracks: the aircraft nearest the chosen one

In [ ]:
from stonesoup.types.mixture import GaussianMixture
from stonesoup.types.numeric import Probability
from stonesoup.types.state import TaggedWeightedGaussianState
from stonesoup.types.update import GaussianMixtureUpdate

try:
    from stonesoup.dataassociator.mfa import MFADataAssociator
    from stonesoup.hypothesiser.mfa import MFAHypothesiser
except ImportError as exc:
    print(f"MFA skipped: {exc}")
else:
    # Seed one track per aircraft, nearest the chosen one first, after MIN_POINTS detections.
    centre, _, _ = data.truth.segment(AIRCRAFT).evaluate([part.start.timestamp()])
    candidates = {}
    for seg in part.looks:
        dets = part.detections_of(seg)
        if len(dets) >= MIN_POINTS:
            xy = part.plot_positions(dets[:1])[0]
            candidates[seg] = (np.hypot(*(xy - centre[0, :2])), dets)
    seeded = sorted(candidates, key=lambda seg: candidates[seg][0])[:MFA_MAX_AIRCRAFT]
    scan_ends = [s.end for s in part.scans]
    mfa_tracks, born = [], {}
    for seg in seeded:
        dets = candidates[seg][1][:MIN_POINTS]
        g = run_filter(ekf_predictor, ekf_updater, make_prior(dets[0]), dets[1:]).state
        component = TaggedWeightedGaussianState(
            g.state_vector, g.covar, timestamp=g.timestamp, weight=Probability(1), tag=[]
        )
        track = Track([GaussianMixture([component])], id=seg)
        mfa_tracks.append(track)
        born[seg] = next(end for end in scan_ends if end > g.timestamp)  # joins at the next scan

    mfa = MFADataAssociator(MFAHypothesiser(pda_hypothesiser), slide_window=SLIDE_WINDOW)
    started = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for scan_end, batch in part.batches():
            live = {t for t in mfa_tracks if born[t.id] < scan_end}
            if not live:
                continue
            hypotheses = mfa.associate(live, batch, scan_end)
            for track in live:
                components = [
                    ekf_updater.update(h) if h else h.prediction for h in hypotheses[track]
                ]
                track.append(
                    GaussianMixtureUpdate(components=components, hypothesis=hypotheses[track])
                )
    seconds = time.perf_counter() - started
    trackers["MFA (seeded)"] = (mfa_tracks, seconds, (0, 2, 4))
    print(f"MFA: {len(mfa_tracks)} seeded tracks in {seconds:.1f} s")

## 13. Comparison

Every tracker scored on the same subset with the same metrics as
`tracking-sim run` (OSPA and GOSPA per scan, SIAP for completeness,
ambiguity and spuriousness; see notebook 03 for how to read the table).
For a loaded run, its own trackers' tracks are scored here too, over the
same subset.

Then the chosen aircraft's replay, with the track from each tracker that
followed it longest. With only a few aircraft near each other, nearest
neighbour and global nearest neighbour make the same choices, and PDA and
JPDA nearly so; the differences show in dense traffic (try your own
capture, or `SUBSET_RADIUS_KM = None`).

In [ ]:
from tracking_sim.eval.metrics import comparison_table

grid = part.grid()
scores = []
for name, (tracks, seconds, mapping) in trackers.items():
    scores.append(
        part.score(
            name,
            tracks,
            position_mapping=mapping,
            velocity_mapping=tuple(i + 1 for i in mapping),
            elapsed_s=seconds,
            grid=grid,
        )
    )
for tracker_id in data.stored_tracks:
    # The run's tracks cover its whole coverage: score only what falls in the subset.
    scores.append(
        part.score(f"{tracker_id} (stored)", part.clip(data.stored(tracker_id)), grid=grid)
    )
print(comparison_table(scores))
best = min(scores, key=lambda s: s.mean_ospa_m)
print(f"\nlowest mean OSPA: {best.tracker_id} ({best.mean_ospa_m:.0f} m)")

fig, ax = plt.subplots(figsize=(9, 3.4))
for i, score in enumerate(scores[: len(SERIES)]):
    minutes = [(t - part.start).total_seconds() / 60 for t in score.times]
    ax.plot(minutes, score.ospa, color=SERIES[i], lw=1.5, label=score.tracker_id)
ax.set(xlabel="minutes into the subset", ylabel="OSPA (m)", ylim=(0, None))
ax.legend(frameon=False, fontsize=8, ncols=3)
ax.set_title("OSPA per scan (lower is better)", loc="left", fontsize=10)
plt.show()

In [ ]:
following = []
for name, (tracks, _, mapping) in trackers.items():
    mine = tracks_following(tracks, AIRCRAFT)
    if mine:
        following.append(Estimate(name, mine[0], mapping))
    print(f"{name}: {len(mine)} track(s) followed {AIRCRAFT}")
tracker_replay = Replay(part, AIRCRAFT, following[: len(SERIES)])
display(tracker_replay.widget())
print("final scan:", tracker_replay.summary(-1))

## Where next

* Change the radar in section 1 (`SENSOR_CHANGES`): more noise, a lower Pd,
  more false alarms, a slower scan, and watch which filters and associators
  degrade first.
* Load a stored run (`TRACKING_SIM_RUN`) to work on the exact detections a
  `tracking-sim run` saw, and compare your hand-built tracker with its
  trackers in section 13.
* Simulate on your own traffic (`TRACKING_SIM_RECORDING`): real flight paths
  have the manoeuvres synthetic ones lack.